# Model owner

You hold a private LoRA adapter for gemma-4-31B-it. This notebook verifies the enclave, uploads the adapter over the attestation-pinned channel, approves the run and collects the receipt. You never see the benchmark owner's prompts or the results.

Setup once: install [uv](https://docs.astral.sh/uv/getting-started/installation/) and the [`tinfoil` CLI](https://docs.tinfoil.sh/containers/cli), then from the repo root run `uv run --group notebooks jupyter notebook notebooks/`. Create your key with `dbe keygen --party model-owner` and have the operator put the printed public key into `tinfoil-config.yml`. Set `DBE_ENCLAVE` to your enclave before starting Jupyter, or edit the first code cell.

In [ ]:
PARTY = "model-owner"
import os, subprocess, json
from pathlib import Path
if Path.cwd().name == "notebooks":       # run from the repo root so bench/ paths resolve
    os.chdir("..")
ENCLAVE = os.environ.get("DBE_ENCLAVE", "dbe.tinfoil.containers.tinfoil.dev")
REPO = os.environ.get("DBE_REPO", "tinfoilsh/double-blind-eval")
TAG = os.environ.get("DBE_TAG", "v0.1.0")
os.environ.update(DBE_ENCLAVE=ENCLAVE, DBE_REPO=REPO, DBE_PARTY=PARTY)

def dbe(*args):
    """Run a dbe command and print its output."""
    proc = subprocess.run(["dbe", *args], capture_output=True, text=True)
    print(proc.stdout or proc.stderr)
    return proc

## 1. Verify the enclave

`dbe verify` checks the Sigstore-published measurement of the release against the enclave's live hardware attestation and pins the TLS key. Everything below refuses to talk to anything else.

In [ ]:
dbe("verify")

## 2. Upload the private adapter

A PEFT adapter directory (`adapter_config.json` + `adapter_model.safetensors`) or a `.tar.gz` of one. It is held in enclave memory and loaded into vLLM; it never touches the host disk. The cell below fetches a public LoRA adapter for gemma-4-31B-it to stand in for a private one; point `DBE_ADAPTER` at your own directory instead if you have one.

In [ ]:
ADAPTER_PATH = os.environ.get("DBE_ADAPTER", "demo-adapter")
if not os.path.exists(os.path.join(ADAPTER_PATH, "adapter_config.json")):
    subprocess.run(["bash", "bench/fetch_demo_adapter.sh", ADAPTER_PATH], check=True)
dbe("model", "hash", ADAPTER_PATH)
dbe("model", "upload", ADAPTER_PATH)

## 3. Check that both uploads are in

Approvals come last. `dbe status` shows whether the benchmark owner's prompt set has landed and who has approved so far. Re-run this cell until both uploads show `[x]`.

In [ ]:
dbe("status")

## 4. Review and approve the run manifest

The manifest names the adapter hash, the benchmark hash, the sampling parameters and the output policy. Compare `manifest_sha256` with the benchmark owner out of band, then sign it. Either party may approve first; the run starts the moment the second approval lands. `dbe approve` refuses to run before both uploads are in, and any re-upload afterwards drops both approvals.

In [ ]:
dbe("manifest")

In [ ]:
dbe("approve")

## 5. Wait for the run and collect the receipt

The output policy gives the model owner the receipt only: proof of what ran, signed by the enclave.

In [ ]:
dbe("run", "--wait")
dbe("receipt", "get", "--out", "receipt.json")
dbe("receipt", "verify", "receipt.json", "--tag", TAG)